<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 11 — A Worked Example: 900,000 Tips and Their Businesses

Notebook 01 counted comparisons on lists of eight names. This notebook does
one ordinary data task on real files and lets the clock show what the counting
predicted.

**The task.** Yelp users leave short *tips* about businesses: "Get the
beignets." Each tip carries the id of its business and nothing else about it.
To say anything about a tip (which city, how many stars) you have to find its
business. There are 908,915 tips and 150,346 businesses.

**The question:** *For every tip, find its business. How long does that
take?*

It is a join, the same one as in Sessions 7 and 8. Here it is done by hand,
three ways, so that you can see what `merge` has been doing for you.

| Step | What happens |
|---|---|
| 1 | Load the two files |
| 2 | The obvious way: scan the list for each tip |
| 3 | Sort the ids once, then halve |
| 4 | A dictionary |
| 5 | What only a set does well: membership |
| 6 | What only sorted data does well: ranges and prefixes |
| 7 | Counting and grouping are hashing too |

> **Run this on ARC, with the kernel set to Conda Python 3.12.** The files are
> under `/data/public/yelp/` and exist only there. Loading them takes a few
> seconds and several hundred megabytes of memory.

In [ ]:
import bisect
import json
import time
from collections import Counter

BUSINESSES = "/data/public/yelp/yelp_academic_dataset_business.json"
TIPS = "/data/public/yelp/yelp_academic_dataset_tip.json"

---

## Step 1 — Load

Both files are JSON Lines: one record per line (Session 8). Only a few fields
of each are kept.

In [ ]:
def read_records(path, fields):
    """Read a JSON Lines file; keep only the named fields of each record."""
    records = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            records.append({field: record[field] for field in fields})
    return records


start = time.perf_counter()
businesses = read_records(BUSINESSES, ["business_id", "name", "city", "stars"])
tips = read_records(TIPS, ["business_id", "user_id", "date"])
print("{:,} businesses and {:,} tips, loaded in {:.1f} s".format(
    len(businesses), len(tips), time.perf_counter() - start))

print(businesses[0])
print(tips[0])

The business ids are 22-character strings. They are in no useful order in
either file.

In [ ]:
ids = [business["business_id"] for business in businesses]
print(ids[:4])
print("all different:", len(set(ids)) == len(ids))

---

## Step 2 — The obvious way

For each tip, find the position of its business id in the list of ids.
`list.index` is a linear search, written in C and as fast as a linear search
can be. Do not run it on all the tips. Try 200.

In [ ]:
sample = tips[:200]

start = time.perf_counter()
positions = [ids.index(tip["business_id"]) for tip in sample]
linear_seconds = time.perf_counter() - start

per_tip = linear_seconds / len(sample)
print("200 tips: {:.2f} s, so {:.1f} milliseconds per tip".format(linear_seconds, 1000 * per_tip))
print("all {:,} tips would take about {:.0f} minutes".format(len(tips), per_tip * len(tips) / 60))

About a millisecond per tip sounds like nothing. Times 908,915 it is around a
quarter of an hour, for one join, with the fastest linear search there is.
The count says why: each lookup compares about 75,000 ids, half the list, so
the whole job is roughly 68 billion comparisons.

And it scales badly in both directions. Twice the tips, twice the time; twice
the businesses, twice again.

Check that the 200 answers are right before going on. They will be the
reference for the faster methods.

In [ ]:
for tip, position in zip(sample[:3], positions[:3]):
    print(tip["business_id"], "->", businesses[position]["name"], "in", businesses[position]["city"])

assert all(ids[position] == tip["business_id"] for tip, position in zip(sample, positions))

---

## Step 3 — Sort once, then halve

Binary search needs the ids in order. Sorting the list of ids alone would lose
the link to the businesses, so sort the **positions** by the id they point to:
`order[0]` is the position of the business with the smallest id.

In [ ]:
start = time.perf_counter()
order = sorted(range(len(ids)), key=lambda position: ids[position])
sorted_ids = [ids[position] for position in order]
sort_seconds = time.perf_counter() - start

print("sorted {:,} ids in {:.2f} s".format(len(ids), sort_seconds))
print(sorted_ids[:3])

In [ ]:
def find_sorted(business_id):
    """Return the position of a business in `businesses`, or -1."""
    place = bisect.bisect_left(sorted_ids, business_id)
    if place < len(sorted_ids) and sorted_ids[place] == business_id:
        return order[place]
    return -1


assert [find_sorted(tip["business_id"]) for tip in sample] == positions

start = time.perf_counter()
found = [find_sorted(tip["business_id"]) for tip in tips]
binary_seconds = time.perf_counter() - start

print("all {:,} tips: {:.2f} s".format(len(tips), binary_seconds))
print("not found:", found.count(-1))

The same answers as the linear search for the 200, and all 908,915 tips done
in under a second. About 17 comparisons per tip in place of 75,000.

`bisect_left` returns where the id *would* go, whether or not it is there.
The two lines after it check that it really is: forgetting that check is the
usual bug with `bisect`.

The sort cost a fraction of a second, once. It paid for itself within the
first few hundred lookups.

---

## Step 4 — A dictionary

A dictionary from id to position: a hash table. Building it is one pass over
the businesses.

In [ ]:
start = time.perf_counter()
position_of = {business_id: position for position, business_id in enumerate(ids)}
build_seconds = time.perf_counter() - start

assert [position_of[tip["business_id"]] for tip in sample] == positions

start = time.perf_counter()
found = [position_of.get(tip["business_id"], -1) for tip in tips]
hash_seconds = time.perf_counter() - start

print("built the dictionary in {:.2f} s".format(build_seconds))
print("all {:,} tips: {:.2f} s".format(len(tips), hash_seconds))

In [ ]:
print("{:28} {:>12}".format("method", "all the tips"))
print("{:28} {:>9.0f} min".format("scan the list (estimated)", per_tip * len(tips) / 60))
print("{:28} {:>10.2f} s  (plus {:.2f} s to sort)".format("sort, then bisect", binary_seconds, sort_seconds))
print("{:28} {:>10.2f} s  (plus {:.2f} s to build)".format("dictionary", hash_seconds, build_seconds))

A quarter of an hour, under a second, a quarter of a second. The step from the
first to the second is the important one: from *unusable* to *instant*. The
step from the second to the third is a factor of three, which is real, and
nobody would notice it.

That is the usual shape. The big win is getting rid of the linear search
inside the loop. Which of the two fast methods replaces it matters much less,
and is decided by what else you need (Steps 5 and 6).

This is also what `pandas.merge` does: it builds a hash table of one table's
keys and looks the other table's keys up in it.

**✏️ Try it:** Use `position_of` to print, for the first five tips, the business name, its city and its stars.

---

## Step 5 — What a set does well: membership

Three questions about *whether*, not *where*.

**Which businesses have no tip at all?**

In [ ]:
tipped = {tip["business_id"] for tip in tips}             # a set: each id once
untipped = [business for business in businesses if business["business_id"] not in tipped]

print("{:,} businesses have at least one tip".format(len(tipped)))
print("{:,} have none, for example {!r}".format(len(untipped), untipped[0]["name"]))

`not in tipped` is one step per business because `tipped` is a set. Written
with a list (`not in [tip["business_id"] for tip in tips]`) the same line
would scan up to 908,915 tips for each of 150,346 businesses.

**Does every tip refer to a business we have?**

In [ ]:
known = set(ids)
orphans = [tip for tip in tips if tip["business_id"] not in known]
print("tips whose business is missing:", len(orphans))

None. This is the orphan check of Homework 6, and with a set it is cheap
enough to run every time, before trusting a join.

**How many different people wrote tips?** Set arithmetic answers "how many
different" directly.

In [ ]:
writers = {tip["user_id"] for tip in tips}
print("{:,} tips by {:,} different users".format(len(tips), len(writers)))

---

## Step 6 — What sorted data does well: ranges and prefixes

A hash table answers "is exactly this key here?" and nothing else. Two
questions it cannot help with:

**How many businesses have a name that starts with "Star"?** In sorted order
they sit next to each other. Find where "Star" would go, and where "Stas", the
first thing after every "Star...", would go.

In [ ]:
names = sorted(business["name"] for business in businesses)

first = bisect.bisect_left(names, "Star")
past = bisect.bisect_left(names, "Stas")

print("{:,} names start with 'Star', from {!r} to {!r}".format(
    past - first, names[first], names[past - 1]))
print("of which {:,} are exactly 'Starbucks'".format(
    bisect.bisect_right(names, "Starbucks") - bisect.bisect_left(names, "Starbucks")))

Two searches of about 17 steps each, and the answer is a subtraction. The 882
names in between were never looked at.

**How many tips were written in 2019?** The dates are text like
`2019-07-04 18:22:10`, which sorts correctly as text because the year comes
first.

In [ ]:
dates = sorted(tip["date"] for tip in tips)
print("from", dates[0], "to", dates[-1])

in_2019 = bisect.bisect_left(dates, "2020-01-01") - bisect.bisect_left(dates, "2019-01-01")
print("{:,} tips in 2019".format(in_2019))

In [ ]:
for year in range(2009, 2023):
    count = (bisect.bisect_left(dates, str(year + 1) + "-01-01")
             - bisect.bisect_left(dates, str(year) + "-01-01"))
    print(year, "{:>8,}".format(count), "#" * (count // 4000))

Fourteen range counts over 908,915 dates, each from two short searches. With a
dictionary keyed by date you could ask only for one exact second at a time.

Order is what sorted data has and a hash table throws away. *Between*,
*starts with*, *the nearest*, *the next one* are all order questions.

**✏️ Try it:** How many tips were written in July 2014? And how many business names start with a digit (from '0' up to, not including, ':')?

---

## Step 7 — Counting and grouping are hashing too

**Which businesses get the most tips?** Counting means: for each tip, find the
tally for its business and add one. That is a lookup per tip, so the tallies
live in a dictionary.

In [ ]:
tally = {}
for tip in tips:
    business_id = tip["business_id"]
    tally[business_id] = tally.get(business_id, 0) + 1

top = sorted(tally.items(), key=lambda item: item[1], reverse=True)[:5]
for business_id, count in top:
    business = businesses[position_of[business_id]]
    print("{:>6,}  {} ({})".format(count, business["name"], business["city"]))

This is the loop-and-dictionary pattern from Session 3, and now you know why
it is fast: `tally.get` and `tally[...] =` are each one step, so the whole
count is one pass over the tips. `collections.Counter` is the same thing,
packaged:

In [ ]:
assert Counter(tip["business_id"] for tip in tips) == tally
print(Counter(tally.values()).most_common(3), "<- (tips per business, how many businesses)")

Most businesses with any tips have one or two, and a beignet café in New
Orleans has 2,571. Session 9 would call that skewed.

`groupby` in pandas is this pattern as well: a hash table from each group's
key to its rows.

### What the three methods were good for

| Question | Method | Why |
|---|---|---|
| Find the business of each tip | dictionary | exact key, many lookups |
| Which businesses have no tips; which tips have no business | set | membership |
| How many different users | set | duplicates vanish |
| Names starting with "Star"; tips in 2019 | sorted list and `bisect` | order |
| Tips per business | dictionary | a tally per key |
| A single lookup in a short list | list and `in` | nothing else is worth setting up |

The linear search was the right tool for none of the large jobs, and it is the
one everybody writes first, because `in` on a list looks exactly like `in` on
a set.

---

## Exercises

### 🧩 Exercise 1 — Measure the growth

Time `ids[:n].index(...)` for 200 lookups with `n` = 10,000, 20,000, 40,000
and 80,000 (look up ids that are in that part of the list). Does the time
double when `n` doubles? Do the same with a dictionary built from `ids[:n]`.

### 🧩 Exercise 2 — The join, finished

Using `position_of`, compute the average `stars` of the business of each tip:
that is, the mean over all 908,915 tips of the stars of the tipped business.
Compare it with the plain mean of `stars` over the businesses. Why do they
differ?

### 🧩 Exercise 3 — Cities by prefix

Sort the distinct city names. How many start with "Saint" and how many with
"St"? Use `bisect` for both, and print the matching names to check.

### 🧩 Exercise 4 — Busiest day

Which single day had the most tips? Decide first which structure fits: a
count per day is a tally, so a dictionary keyed by `tip["date"][:10]`.

### 🧩 Exercise 5 — Predict, then run

Somebody writes `if tip["user_id"] in user_ids:` inside a loop over all the
tips, where `user_ids` is a **list** of the 301,758 distinct users. Estimate
from Step 2 how long the loop would take. Then say what one-line change fixes
it, make the change, and time it.

---

**Next:** `03_Search_and_Hashing_Reference` lists the tools used here, by
task. In Homework 10 you build all three methods yourself and count their
comparisons on the same business ids.